In [ ]:
from ddi.data import load_brat_docs
docs = load_brat_docs("Train")
print(len(docs))
print(docs[0].register, len(docs[0].entities), len(docs[0].relations))

In [ ]:
from ddi.data import make_sentence_level

import spacy
nlp = spacy.load("en_core_web_sm")

docs = load_brat_docs("Train")
sent_docs = make_sentence_level(docs[0], nlp)
print(len(sent_docs), "sentences from doc 0")
print(sent_docs[0].text)
print(sent_docs[0].register)
for e in sent_docs[0].entities:
    print(e.type, e.text, e.locations.begin(), e.locations.end())

In [ ]:
# 1. What functions does the file ACTUALLY define, on disk right now?
import ddi.data, importlib
importlib.reload(ddi.data)
print([n for n in dir(ddi.data) if not n.startswith('_')])

# 2. Which file is Python actually reading?
print(ddi.data.__file__)

In [ ]:
from ddi.data import make_pair_instances

docs = load_brat_docs("Train")
# find a sentence with at least one relation
for doc in docs[:20]:
    for sd in make_sentence_level(doc, nlp):
        if sd.relations:
            insts = make_pair_instances(sd)
            labels = [i['label'] for i in insts]
            if any(l != 'NONE' for l in labels):
                print(sd.text)
                for inst in insts:
                    if inst['label'] != 'NONE':
                        print(inst['label'], "::", inst['text'])
                break
    else:
        continue
    break

In [ ]:
docs = load_brat_docs("Train")
sds = make_sentence_level(docs[0], nlp)
print("doc 0 ->", len(sds), "sentence-docs")
for i, sd in enumerate(sds):
    print(i, "| rels:", len(sd.relations), "| ents:", len(sd.entities), "|", repr(sd.text))

In [ ]:
for doc in load_brat_docs("Train")[:50]:
       if doc.text[:30] in doc.text[30:]:
           print("possible dup:", doc.text)

In [ ]:
from ddi.data import build_human
train, val = build_human()
print(len(train), "train pairs |", len(val), "val pairs")

from collections import Counter
print("train labels:", Counter(i['label'] for i in train))
print("val labels:  ", Counter(i['label'] for i in val))
print("train source:", Counter(i['source'] for i in train))

In [ ]:
from ddi.data import build_human, downsample_train_negatives

train, val = build_human()
for r in [5, 9, None]:
    d = downsample_train_negatives(train, r)
    n_pos = sum(1 for x in d if x['label'] != 'NONE')
    print(f"ratio={r}: {len(d)} instances, {n_pos} pos, {len(d)-n_pos} neg")